## CIC-DoH-Brw-2020: data loading, `full_graph_process`, and binary trainers

Loads `l2-benign.csv` / `l2-malicious.csv` from `cic-dataset/CIRA-CIC-DoHBrw-2020`, concatenates them, applies a **stratified** train/test split (configurable `TEST_SIZE`) so class **imbalance** is preserved in both splits, then trains `BasicGNNClassifier` and `BasicSGNNClassifier` using `sgnn.trainers.bin_class_train`.

`BasicSGNNClassifier` uses a fixed `SrcDstGraph` inside `time_chunk_x`; when graphs differ, we train one SGNN per graph (batch size 1). `BasicGNNClassifier` is trained on both class graphs in one dataset.

The classification report cell evaluates on **train** and **held-out test** graphs.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
import torch

# Parent of this folder (`snn-stuff`) on sys.path so `import sgnn` works when cwd is `sgnn/`
ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from sgnn.graph_builder.graph_builder import SrcDstGraph
from sgnn.graph_builder.graph_custom_data import (
    GraphCustomData,
    GraphCustomDataLoader,
    graphs_from_src_dst_list,
)
from sgnn.models.gnn import BasicGNNClassifier
from sgnn.models.sgnn import BasicSGNNClassifier
from sgnn.trainers.bin_class_train import (
    GraphBinaryClassificationDataset,
    train_basic_gnn_classifier_binary_nadam,
    train_basic_gnn_classifier_binary_sgd,
    train_basic_sgnn_classifier_binary_nadam,
    train_basic_sgnn_classifier_binary_sgd,
)


/Users/smurli/local_dev/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
%pip install torch snntorch pandas torch-geometric networkx scikit-learn


Note: you may need to restart the kernel to use updated packages.


In [3]:
DATA_DIR = ROOT / "cic-dataset" / "CIRA-CIC-DoHBrw-2020"

# Train/test split: fraction of rows held out for testing. Stratified split keeps the
# benign vs malicious imbalance similar in train and test.
TEST_SIZE = 0.2
RANDOM_STATE = 42

df_benign = pd.read_csv(DATA_DIR / "l2-benign.csv")
df_malicious = pd.read_csv(DATA_DIR / "l2-malicious.csv")

df_benign["binary_label"] = 0
df_malicious["binary_label"] = 1

df_all = pd.concat([df_benign, df_malicious], ignore_index=True)


def replace_nan_in_flow_features(df: pd.DataFrame, feature_cols: list[str]) -> pd.DataFrame:
    """Coerce feature columns to numeric and replace NaN/inf with 0.0 before graph build."""
    out = df.copy()
    replaced = 0
    for col in feature_cols:
        if col not in out.columns:
            continue
        s = pd.to_numeric(out[col], errors="coerce")
        n = int(s.isna().sum())
        replaced += n
        out[col] = s.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    if replaced:
        print(
            f"Sanitized flow features: coerced to numeric and filled {replaced} NaN cell(s) with 0.0 "
            f"(columns in {len(feature_cols)}-feature list)."
        )
    return out


import numpy as np

df_all = replace_nan_in_flow_features(df_all, SrcDstGraph.DEFAULT_FEATURES)

from sklearn.model_selection import train_test_split

df_train, df_test = train_test_split(
    df_all,
    test_size=TEST_SIZE,
    stratify=df_all["binary_label"],
    random_state=RANDOM_STATE,
)

df_benign_train = df_train[df_train["binary_label"] == 0].copy()
df_malicious_train = df_train[df_train["binary_label"] == 1].copy()
df_benign_test = df_test[df_test["binary_label"] == 0].copy()
df_malicious_test = df_test[df_test["binary_label"] == 1].copy()

n0, n1 = int((df_all["binary_label"] == 0).sum()), int((df_all["binary_label"] == 1).sum())
print(
    f"Full dataset: {len(df_all)} rows — benign={n0}, malicious={n1} "
    f"(imbalance ratio {n0 / max(n1, 1):.3f} : 1)"
)
print(
    f"Train ({100 * (1 - TEST_SIZE):.0f}%): {len(df_train)} rows — "
    f"benign={len(df_benign_train)}, malicious={len(df_malicious_train)}"
)
print(
    f"Test ({100 * TEST_SIZE:.0f}%):  {len(df_test)} rows — "
    f"benign={len(df_benign_test)}, malicious={len(df_malicious_test)}"
)
df_train.head()


Sanitized flow features: coerced to numeric and filled 688 NaN cell(s) with 0.0 (columns in 29-feature list).
Full dataset: 269643 rows — benign=19807, malicious=249836 (imbalance ratio 0.079 : 1)
Train (80%): 215714 rows — benign=15846, malicious=199868
Test (20%):  53929 rows — benign=3961, malicious=49968


,SourceIP,DestinationIP,SourcePort,DestinationPort,TimeStamp,Duration,FlowBytesSent,FlowSentRate,FlowBytesReceived,FlowReceivedRate,...,ResponseTimeTimeVariance,ResponseTimeTimeStandardDeviation,ResponseTimeTimeMean,ResponseTimeTimeMedian,ResponseTimeTimeMode,ResponseTimeTimeSkewFromMedian,ResponseTimeTimeSkewFromMode,ResponseTimeTimeCoefficientofVariation,Label,binary_label
23136,192.168.20.205,9.9.9.11,34822,443,2020-03-31 04:07:14,33.674378,1807,53.660976,4827,143.343405,...,0.000052,0.007225,0.010324,0.015273,0.000058,-2.055075,1.420799,0.699875,Malicious,1
130624,192.168.20.144,9.9.9.11,42838,443,2020-03-31 16:53:08,33.783112,1875,55.501104,4965,146.966922,...,0.000146,0.012099,0.011992,0.015268,0.000004,-0.812177,0.990832,1.008917,Malicious,1
13358,192.168.20.111,8.8.8.8,38788,443,2019-12-19 21:38:53,3.184600,1178,369.905169,1629,511.524210,...,0.000131,0.011461,0.011314,0.006037,0.000014,1.381312,0.985938,1.013008,Benign,0
252395,192.168.20.211,9.9.9.11,57340,443,2020-03-31 16:56:40,33.543746,1875,55.897156,4897,145.988465,...,0.000057,0.007557,0.009291,0.015321,0.000005,-2.393779,1.228881,0.813311,Malicious,1
115208,176.103.130.130,192.168.20.144,443,40994,2020-03-31 21:38:38,119.474792,873,7.306981,772,6.461614,...,47.735718,6.909104,10.554037,15.077291,0.000005,-1.964041,1.527554,0.654641,Malicious,1


In [4]:
FEATURES = SrcDstGraph.DEFAULT_FEATURES

# Each call to full_graph_process fills graph_ls with one SrcDstGraph per dataframe row.
_probe = SrcDstGraph()
_probe.full_graph_process(df_train.head(3), FEATURES)
print(
    "Per-flow graphs: len(graph_ls) matches row count;",
    "sample run on 3 train rows ->",
    len(_probe.graph_ls),
    "graphs",
)


Per-flow graphs: len(graph_ls) matches row count; sample run on 3 train rows -> 3 graphs


In [ ]:
# Optional caps for faster runs (None = all rows in each split)
MAX_TRAIN_ROWS_PER_CLASS = None
MAX_TEST_ROWS_PER_CLASS = None


def _limit_df(df: pd.DataFrame, n: int | None) -> pd.DataFrame:
    if n is None or len(df) <= n:
        return df
    return df.head(n).copy()


dfb_tr = _limit_df(df_benign_train, MAX_TRAIN_ROWS_PER_CLASS)
dfm_tr = _limit_df(df_malicious_train, MAX_TRAIN_ROWS_PER_CLASS)
dfb_te = _limit_df(df_benign_test, MAX_TEST_ROWS_PER_CLASS)
dfm_te = _limit_df(df_malicious_test, MAX_TEST_ROWS_PER_CLASS)

build_b = SrcDstGraph()
build_b.full_graph_process(dfb_tr, FEATURES, max_workers = 1)
print("Training graphs processed")
build_m = SrcDstGraph()
build_m.full_graph_process(dfm_tr.sample(n= 30000), FEATURES, max_workers = 1)
print("Training graphs processed")

pyg_train = graphs_from_src_dst_list(build_b.graph_ls + build_m.graph_ls, time_chunked=False)
labels_train = [0] * len(build_b.graph_ls) + [1] * len(build_m.graph_ls)
src_train = build_b.graph_ls + build_m.graph_ls
train_ds = GraphBinaryClassificationDataset(
    pyg_train, labels=labels_train, src_dst_graphs=src_train
)


full_graph_process (max_workers=1): 10000 cumulative graphs built after 10000 rows
Training graphs processed
full_graph_process (max_workers=1): 10000 cumulative graphs built after 10000 rows
full_graph_process (max_workers=1): 20000 cumulative graphs built after 20000 rows
full_graph_process (max_workers=1): 30000 cumulative graphs built after 30000 rows
Training graphs processed
Test graphs processed
full_graph_process (max_workers=1): 10000 cumulative graphs built after 10000 rows
full_graph_process (max_workers=1): 20000 cumulative graphs built after 20000 rows
Test graphs processed
Train graphs: 45846 (benign 15846 + malicious 30000 )  |  Test graphs: 23961


In [22]:
build_bt = SrcDstGraph()
build_bt.full_graph_process(dfb_te, FEATURES, max_workers = 1)
print("Test graphs processed")
build_mt = SrcDstGraph()
build_mt.full_graph_process(dfm_te.sample(n = 20000), FEATURES, max_workers = 1)
print("Test graphs processed")
pyg_test = graphs_from_src_dst_list(build_bt.graph_ls + build_mt.graph_ls, time_chunked=False)
labels_test = [0] * len(build_bt.graph_ls) + [1] * len(build_mt.graph_ls)
src_test = build_bt.graph_ls + build_mt.graph_ls
test_ds = GraphBinaryClassificationDataset(
    pyg_test, labels=labels_test, src_dst_graphs=src_test
)

print(
    "Train graphs:",
    len(train_ds),
    "(benign",
    len(build_b.graph_ls),
    "+ malicious",
    len(build_m.graph_ls),
    ")  |  Test graphs:",
    len(test_ds),
)

Test graphs processed
full_graph_process (max_workers=1): 10000 cumulative graphs built after 10000 rows
full_graph_process (max_workers=1): 20000 cumulative graphs built after 20000 rows
Test graphs processed


NameError: name 'build_b' is not defined

In [14]:
in_channels = pyg_train[0].x.size(1)
assert in_channels == len(FEATURES)
HIDDEN = 32
TIMESTEPS = 10

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS = 15
BATCH = 5000
WEIGHT_DECAY = 1e-6  # L2 regularization via optimizer weight_decay; set 0.0 to disable

print('GNN Training Start')

gnn_nadam = BasicGNNClassifier(
    in_channels=in_channels,
    hidden_channels=HIDDEN,
    num_classes=2,
    num_conv_layers=2,
).to(device)

hist_gnn_nadam = train_basic_gnn_classifier_binary_nadam(
    gnn_nadam,
    train_ds,
    device=device,
    epochs=EPOCHS,
    batch_size=BATCH,
    lr=.001,
    weight_decay=WEIGHT_DECAY*10,
)

gnn_sgd = BasicGNNClassifier(
    in_channels=in_channels,
    hidden_channels=HIDDEN,
    num_classes=2,
    num_conv_layers=2,
).to(device)

hist_gnn_sgd = train_basic_gnn_classifier_binary_sgd(
    gnn_sgd,
    train_ds,
    device=device,
    epochs=EPOCHS,
    batch_size=BATCH,
    lr=.001,
    momentum=0.75,
    weight_decay=WEIGHT_DECAY*10,
)

trained_gnn = {"NAdam": gnn_nadam, "SGD": gnn_sgd}

print('SGNN Training Start')

SNN_HIDDEN = 87
# Template G for layers; each forward uses dataset src_dst_graphs[i] via graph_build=

sgnn_nadam = BasicSGNNClassifier(
    in_channels=in_channels,
    hidden_channels=SNN_HIDDEN,
    num_classes=2,
    G=src_train[0],
    timesteps=TIMESTEPS,
).to(device)
hist_sgnn_nadam = train_basic_sgnn_classifier_binary_nadam(
    sgnn_nadam,
    train_ds,
    device=device,
    epochs=EPOCHS,
    batch_size=1,
    lr=.00005,
    mini_batch = .05,
    weight_decay=WEIGHT_DECAY,
)

sgnn_sgd = BasicSGNNClassifier(
    in_channels=in_channels,
    hidden_channels=SNN_HIDDEN,
    num_classes=2,
    G=src_train[0],
    timesteps=TIMESTEPS,
).to(device)
hist_sgnn_sgd = train_basic_sgnn_classifier_binary_sgd(
    sgnn_sgd,
    train_ds,
    device=device,
    epochs=EPOCHS,
    batch_size=1,
    lr=.00015,
    mini_batch = .05,
    momentum=0.75,
    weight_decay=WEIGHT_DECAY,
)

trained_sgnn = {"NAdam": sgnn_nadam, "SGD": sgnn_sgd}

GNN Training Start
[epoch 1/15] train_loss=0.733789
[epoch 2/15] train_loss=0.669259
[epoch 3/15] train_loss=0.626753
[epoch 4/15] train_loss=0.614215
[epoch 5/15] train_loss=0.604632
[epoch 6/15] train_loss=0.601265
[epoch 7/15] train_loss=0.592962
[epoch 8/15] train_loss=0.590153
[epoch 9/15] train_loss=0.585721
[epoch 10/15] train_loss=0.580291
[epoch 11/15] train_loss=0.567951
[epoch 12/15] train_loss=0.567177
[epoch 13/15] train_loss=0.557980
[epoch 14/15] train_loss=0.552512
[epoch 15/15] train_loss=0.557852
[epoch 1/15] train_loss=0.762452
[epoch 2/15] train_loss=0.724573
[epoch 3/15] train_loss=0.697133
[epoch 4/15] train_loss=0.695994
[epoch 5/15] train_loss=0.694213
[epoch 6/15] train_loss=0.693414
[epoch 7/15] train_loss=0.693150
[epoch 8/15] train_loss=0.693325
[epoch 9/15] train_loss=0.692913
[epoch 10/15] train_loss=0.693671
[epoch 11/15] train_loss=0.692694
[epoch 12/15] train_loss=0.692572
[epoch 13/15] train_loss=0.692153
[epoch 14/15] train_loss=0.692188
[epoch 15/15]

In [ ]:
import numpy as np
from sklearn.metrics import classification_report


@torch.no_grad()
def graph_level_predictions(
    model: torch.nn.Module,
    dataset: GraphBinaryClassificationDataset,
    device: torch.device,
    *,
    eval_batch_size: int = 256,
) -> tuple[np.ndarray, np.ndarray]:
    """Per-graph predictions; batched for BasicGNNClassifier."""
    bs = min(eval_batch_size, max(1, len(dataset)))
    loader = GraphCustomDataLoader(dataset, batch_size=bs, shuffle=False)
    model.eval()
    ys: list[int] = []
    preds: list[int] = []
    for batch in loader:
        batch = batch.to(device)
        logits = model(batch.x, batch.edge_index, batch.batch)
        y = batch.y.view(-1).long()
        pred = logits.argmax(dim=-1)
        ys.extend(y.cpu().tolist())
        preds.extend(pred.cpu().tolist())
    return np.array(ys), np.array(preds)


@torch.no_grad()
def graph_level_predictions_sgnn(
    model: BasicSGNNClassifier,
    dataset: GraphBinaryClassificationDataset,
    device: torch.device,
) -> tuple[np.ndarray, np.ndarray]:
    """Per-graph SGNN with each sample's SrcDstGraph (graph_ls alignment)."""
    if dataset.src_dst_graphs is None:
        raise ValueError("dataset needs src_dst_graphs")
    model.eval()
    ys: list[int] = []
    preds: list[int] = []
    for i in range(len(dataset)):
        data = dataset[i].to(device)
        G = dataset.src_dst_graphs[i]
        bvec = torch.zeros(data.num_nodes, dtype=torch.long, device=device)
        logits = model(data.x, data.edge_index, bvec, graph_build=G)
        y = data.y.view(-1).long()
        pred = logits.argmax(dim=-1)
        ys.extend(y.cpu().tolist())
        preds.extend(pred.cpu().tolist())
    return np.array(ys), np.array(preds)


target_names = ["benign (0)", "malicious (1)"]

for split_name, ds in ("train", train_ds), ("test", test_ds):
    print(f"=== BasicGNNClassifier ({split_name}) ===\n")
    for name, model in trained_gnn.items():
        y_true, y_pred = graph_level_predictions(model, ds, device)
        print(f"--- GNN + {name} @ {split_name} (n={len(y_true)}) ---")
        print(
            classification_report(
                y_true,
                y_pred,
                labels=[0, 1],
                target_names=target_names,
                zero_division=0,
            )
        )

for split_name, ds in ("train", train_ds), ("test", test_ds):
    print(f"=== BasicSGNNClassifier ({split_name}) ===\n")
    for name, model in trained_sgnn.items():
        y_true, y_pred = graph_level_predictions_sgnn(model, ds, device)
        print(f"--- SGNN + {name} @ {split_name} (n={len(y_true)}) ---")
        print(
            classification_report(
                y_true,
                y_pred,
                labels=[0, 1],
                target_names=target_names,
                zero_division=0,
            )
        )


=== BasicGNNClassifier (train) ===

--- GNN + NAdam @ train (n=45846) ---
               precision    recall  f1-score   support

   benign (0)       0.63      0.68      0.65     15846
malicious (1)       0.82      0.79      0.80     30000

     accuracy                           0.75     45846
    macro avg       0.72      0.73      0.73     45846
 weighted avg       0.75      0.75      0.75     45846

--- GNN + SGD @ train (n=45846) ---
               precision    recall  f1-score   support

   benign (0)       0.35      1.00      0.51     15846
malicious (1)       0.00      0.00      0.00     30000

     accuracy                           0.35     45846
    macro avg       0.17      0.50      0.26     45846
 weighted avg       0.12      0.35      0.18     45846

=== BasicGNNClassifier (test) ===

--- GNN + NAdam @ test (n=23961) ---
               precision    recall  f1-score   support

   benign (0)       0.38      0.67      0.49      3961
malicious (1)       0.92      0.78      0

In [ ]:
# Per-class stratified accuracy by graph size (node count), using mean/std-dev bands
# computed separately for benign (0) and malicious (1) graphs.
import numpy as np


def _dataset_node_counts_and_labels(ds) -> tuple[np.ndarray, np.ndarray]:
    node_counts = np.array([ds[i].num_nodes for i in range(len(ds))], dtype=np.float64)
    labels = np.array([int(ds[i].y.item()) for i in range(len(ds))], dtype=np.int64)
    return node_counts, labels


def _class_band_bounds(node_counts: np.ndarray, labels: np.ndarray, class_label: int):
    class_counts = node_counts[labels == class_label]
    mean_nodes = float(class_counts.mean())
    std_nodes = float(class_counts.std())
    bounds = [
        (mean_nodes - 2 * std_nodes, mean_nodes - std_nodes),
        (mean_nodes - std_nodes, mean_nodes),
        (mean_nodes, mean_nodes + std_nodes),
        (mean_nodes + std_nodes, mean_nodes + 2 * std_nodes),
    ]
    return mean_nodes, std_nodes, bounds


def _band_mask(values: np.ndarray, low: float, high: float, *, include_high: bool) -> np.ndarray:
    if include_high:
        return (values >= low) & (values <= high)
    return (values >= low) & (values < high)


def _accuracy_by_class_bins(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    node_counts: np.ndarray,
    *,
    class_label: int,
    class_name: str,
    title: str,
) -> None:
    class_mask = y_true == class_label
    class_counts = node_counts[class_mask]

    mean_nodes, std_nodes, bounds = _class_band_bounds(node_counts, y_true, class_label)
    print(f"=== {title} | {class_name} ===")
    print(f"mean_nodes={mean_nodes:.6g}, std_nodes={std_nodes:.6g}")

    labels_txt = [
        "mean-2*std to mean-std",
        "mean-std to mean",
        "mean to mean+std",
        "mean+std to mean+2*std",
    ]

    covered = np.zeros(class_counts.shape[0], dtype=bool)
    for b, (low, high) in enumerate(bounds):
        include_high = b == 3
        in_band_class = _band_mask(class_counts, low, high, include_high=include_high)
        covered |= in_band_class

        n = int(in_band_class.sum())
        print(f"--- B{b + 1}: {labels_txt[b]} ({low:g} <= n {'<=' if include_high else '<'} {high:g}) — n={n} ---")
        if n == 0:
            print("(no graphs in this bin)\n")
            continue

        yt_b = y_true[class_mask][in_band_class]
        yp_b = y_pred[class_mask][in_band_class]
        acc = float((yp_b == yt_b).mean())
        correct = int((yp_b == yt_b).sum())
        print(f"accuracy: {acc:.6f} ({correct}/{n})\n")

    dropped = int((~covered).sum())
    if dropped:
        print(f"(excluding {dropped} {class_name} graph(s) outside mean±2*std range)")
    print()


for split_name, ds in (("train", train_ds), ("test", test_ds)):
    node_counts, _ = _dataset_node_counts_and_labels(ds)

    print(f"\n{'#' * 72}\n# BasicGNNClassifier per-class bin accuracy @ {split_name}\n")
    for name, model in trained_gnn.items():
        yt, yp = graph_level_predictions(model, ds, device)
        _accuracy_by_class_bins(
            yt,
            yp,
            node_counts,
            class_label=0,
            class_name="benign (0)",
            title=f"GNN + {name} @ {split_name}",
        )
        _accuracy_by_class_bins(
            yt,
            yp,
            node_counts,
            class_label=1,
            class_name="malicious (1)",
            title=f"GNN + {name} @ {split_name}",
        )

for split_name, ds in (("train", train_ds), ("test", test_ds)):
    node_counts, _ = _dataset_node_counts_and_labels(ds)

    print(f"\n{'#' * 72}\n# BasicSGNNClassifier per-class bin accuracy @ {split_name}\n")
    for name, model in trained_sgnn.items():
        yt, yp = graph_level_predictions_sgnn(model, ds, device)
        _accuracy_by_class_bins(
            yt,
            yp,
            node_counts,
            class_label=0,
            class_name="benign (0)",
            title=f"SGNN + {name} @ {split_name}",
        )
        _accuracy_by_class_bins(
            yt,
            yp,
            node_counts,
            class_label=1,
            class_name="malicious (1)",
            title=f"SGNN + {name} @ {split_name}",
        )



########################################################################
# BasicGNNClassifier per-class bin accuracy @ train

=== GNN + NAdam @ train | benign (0) ===
mean_nodes=9.99148, std_nodes=0.141629
--- B1: mean-2*std to mean-std (9.70822 <= n < 9.84985) — n=0 ---
(no graphs in this bin)

--- B2: mean-std to mean (9.84985 <= n < 9.99148) — n=0 ---
(no graphs in this bin)

--- B3: mean to mean+std (9.99148 <= n < 10.1331) — n=15750 ---
accuracy: 0.678857 (10692/15750)

--- B4: mean+std to mean+2*std (10.1331 <= n <= 10.2747) — n=0 ---
(no graphs in this bin)

(excluding 96 benign (0) graph(s) outside mean±2*std range)

=== GNN + NAdam @ train | malicious (1) ===
mean_nodes=15.7264, std_nodes=0.694066
--- B1: mean-2*std to mean-std (14.3383 <= n < 15.0324) — n=725 ---
accuracy: 0.800000 (580/725)

--- B2: mean-std to mean (15.0324 <= n < 15.7264) — n=0 ---
(no graphs in this bin)

--- B3: mean to mean+std (15.7264 <= n < 16.4205) — n=25602 ---
accuracy: 0.790134 (20229/25602)

--

In [20]:
from pathlib import Path
import torch

save_dir = Path("saved_weights")
save_dir.mkdir(parents=True, exist_ok=True)

saved_paths = []

for name, model in trained_gnn.items():
    out_path = save_dir / f"gnn_{name.lower()}_weights.pt"
    torch.save(model.state_dict(), out_path)
    saved_paths.append(out_path)

for name, model in trained_sgnn.items():
    out_path = save_dir / f"sgnn_{name.lower()}_weights.pt"
    torch.save(model.state_dict(), out_path)
    saved_paths.append(out_path)

print("Saved model weights:")
for p in saved_paths:
    print(f"- {p.resolve()}")

Saved model weights:
- /Users/smurli/local_dev/snn-stuff/sgnn/saved_weights/gnn_nadam_weights.pt
- /Users/smurli/local_dev/snn-stuff/sgnn/saved_weights/gnn_sgd_weights.pt
- /Users/smurli/local_dev/snn-stuff/sgnn/saved_weights/sgnn_nadam_weights.pt
- /Users/smurli/local_dev/snn-stuff/sgnn/saved_weights/sgnn_sgd_weights.pt


In [ ]:
# Ablation study: zero one input feature at a time at evaluation (no library edits).
# Measures test accuracy drop vs baseline for trained GNN / SGNN (NAdam checkpoints).
import numpy as np
import pandas as pd
import torch

from sgnn.trainers.bin_class_train import GraphBinaryClassificationDataset


class _FeatureZeroAblatedDataset(torch.utils.data.Dataset):
    """Wraps a graph binary dataset; optionally zeros one column of ``data.x`` per sample."""

    def __init__(self, base: GraphBinaryClassificationDataset, zero_dim: int | None) -> None:
        self._base = base
        self._zero_dim = zero_dim

    def __len__(self) -> int:
        return len(self._base)

    def __getitem__(self, idx: int):
        g = self._base[idx].clone()
        if self._zero_dim is not None:
            g.x = g.x.clone()
            g.x[:, self._zero_dim] = 0.0
        return g

    @property
    def src_dst_graphs(self):
        return self._base.src_dst_graphs


def _subset_graph_dataset(
    ds: GraphBinaryClassificationDataset, max_graphs: int | None
) -> GraphBinaryClassificationDataset:
    if max_graphs is None or len(ds) <= max_graphs:
        return ds
    graphs = [ds[i] for i in range(max_graphs)]
    src = (
        [ds.src_dst_graphs[i] for i in range(max_graphs)]
        if ds.src_dst_graphs is not None
        else None
    )
    return GraphBinaryClassificationDataset(graphs, labels=None, src_dst_graphs=src)


@torch.no_grad()
def _accuracy(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float((y_true == y_pred).mean())


# None = use full test_ds (slower for SGNN per-graph forward)
ABLATION_MAX_GRAPHS = 8500
ABLATION_OPTIMIZER = "SGD"  # key in trained_gnn / trained_sgnn

_ds_eval = _subset_graph_dataset(test_ds, ABLATION_MAX_GRAPHS)
_feature_names = list(FEATURES)
n_feat = len(_feature_names)

print(
    f"Ablation: input feature zeroing on test split "
    f"(n={len(_ds_eval)} graphs"
    f"{'' if ABLATION_MAX_GRAPHS is None else f', capped from {len(test_ds)}'}"
    f"). Optimizer={ABLATION_OPTIMIZER!r}.\n"
)

rows: list[dict] = []

gnn_base = trained_gnn[ABLATION_OPTIMIZER]
sgnn_base = trained_sgnn[ABLATION_OPTIMIZER]
yt_g, yp_g = graph_level_predictions(gnn_base, _ds_eval, device)
acc_g_base = _accuracy(yt_g, yp_g)
yt_s, yp_s = graph_level_predictions_sgnn(sgnn_base, _ds_eval, device)
acc_s_base = _accuracy(yt_s, yp_s)

rows.append(
    {
        "ablated_feature": "(none - baseline)",
        "feature_index": -1,
        "gnn_accuracy": acc_g_base,
        "gnn_delta_vs_baseline": 0.0,
        "sgnn_accuracy": acc_s_base,
        "sgnn_delta_vs_baseline": 0.0,
    }
)

for j in range(n_feat):
    ds_g = _FeatureZeroAblatedDataset(_ds_eval, j)
    yt, yp = graph_level_predictions(gnn_base, ds_g, device)
    acc_g = _accuracy(yt, yp)

    ds_s = _FeatureZeroAblatedDataset(_ds_eval, j)
    yt, yp = graph_level_predictions_sgnn(sgnn_base, ds_s, device)
    acc_s = _accuracy(yt, yp)

    rows.append(
        {
            "ablated_feature": _feature_names[j],
            "feature_index": j,
            "gnn_accuracy": acc_g,
            "gnn_delta_vs_baseline": acc_g_base - acc_g,
            "sgnn_accuracy": acc_s,
            "sgnn_delta_vs_baseline": acc_s_base - acc_s,
        }
    )

abl_df = pd.DataFrame(rows)
abl_df = abl_df.sort_values("gnn_delta_vs_baseline", ascending=False)

pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 40)

print("Sorted by GNN accuracy drop (larger delta = more impact under this probe):\n")
try:
    display(abl_df)
except NameError:
    print(abl_df.to_string(index=False))

print(
    "\nTip: set ABLATION_MAX_GRAPHS = None for full test set, or switch ABLATION_OPTIMIZER "
    "to 'SGD' to compare optimizers."
)


Ablation: input feature zeroing on test split (n=5000 graphs, capped from 23961). Optimizer='SGD'.

Sorted by GNN accuracy drop (larger delta = more impact under this probe):



,ablated_feature,feature_index,gnn_accuracy,gnn_delta_vs_baseline,sgnn_accuracy,sgnn_delta_vs_baseline
0,(none - baseline),-1,0.7922,0.0,0.7714,0.0000
1,Duration,0,0.7922,0.0,0.7714,0.0000
28,ResponseTimeTimeSkewFromMode,27,0.7922,0.0,0.7714,0.0000
27,ResponseTimeTimeSkewFromMedian,26,0.7922,0.0,0.7714,0.0000
26,ResponseTimeTimeMode,25,0.7922,0.0,0.7714,0.0000
25,ResponseTimeTimeMedian,24,0.7922,0.0,0.7714,0.0000
24,ResponseTimeTimeMean,23,0.7922,0.0,0.7714,0.0000
23,ResponseTimeTimeStandardDeviation,22,0.7922,0.0,0.7714,0.0000
22,ResponseTimeTimeVariance,21,0.7922,0.0,0.7714,0.0000
21,PacketTimeCoefficientofVariation,20,0.7922,0.0,0.7714,0.0000



Tip: set ABLATION_MAX_GRAPHS = None for full test set, or switch ABLATION_OPTIMIZER to 'SGD' to compare optimizers.
